In [ ]:
import numpy as np
import  pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tabpfn import TabPFNRegressor
import math

def  mape(y_true, y_pred):
    return np.mean(np.abs((y_pred - y_true) / y_true)) * 100



import warnings
warnings.filterwarnings("ignore")

plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['font.family'] = 'Times New Roman'  
plt.rcParams['axes.unicode_minus'] = False  
plt.rcParams['font.serif'] = ['Times New Roman'] 


In [ ]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from tabpfn import TabPFNRegressor


def mape(y_pred, y_true):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100


def create_sliding_window_with_y(X, y, window_size=5, forward_step=1):
    X_new = []
    y_new = []
    columns = []

    for lag in range(window_size, 0, -1):
        for col in X.columns:
            columns.append(f"{col}_t-{lag}")

    for lag in range(window_size, 0, -1):
        columns.append(f"y_t-{lag}")

    for i in range(window_size, len(X) - forward_step + 1):

        X_window = X.iloc[i-window_size:i].values.flatten()
        y_window = y.iloc[i-window_size:i].values.flatten()

        features = np.concatenate([X_window, y_window])

        X_new.append(features)

        y_new.append(y.iloc[i + forward_step - 1])

    X_new = pd.DataFrame(X_new, columns=columns)
    y_new = pd.Series(y_new)

    return X_new, y_new


df_data = pd.read_csv(
    'data.csv',
    sep=',',
    header=0,
    index_col=0
)

original_index = df_data.index

X_raw = df_data.iloc[:, :-5]
y_raw = df_data.iloc[:, -1]

window_size = 5
forward_step = 1   # predictive time length
X, y = create_sliding_window_with_y(X_raw, y_raw, window_size, forward_step)


train_size = int(len(X) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]
y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

train_time_index = original_index[window_size:train_size + window_size]
test_time_index = original_index[train_size + window_size:train_size + window_size + len(y_test)]

scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_x.fit_transform(X_train)
X_test = scaler_x.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

reg = TabPFNRegressor(ignore_pretraining_limits=True)
reg.fit(X_train, y_train_scaled)

y_train_pred_scaled = reg.predict(X_train)
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).ravel()

print('MSE:', mean_squared_error(y_train, y_train_pred))
print('RMSE:', np.sqrt(mean_squared_error(y_train, y_train_pred)))
print('MAE:', mean_absolute_error(y_train, y_train_pred))
print('MAPE:', mape(y_train_pred, y_train))
print('R2:', r2_score(y_train, y_train_pred))

y_test_pred_scaled = reg.predict(X_test)
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()


print('MSE:', mean_squared_error(y_test, y_test_pred))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_test_pred)))
print('MAE:', mean_absolute_error(y_test, y_test_pred))
print('MAPE:', mape(y_test_pred, y_test))
print('R2:', r2_score(y_test, y_test_pred))

threshold = 25
y_train_class = (y_train >= threshold).astype(int)
y_train_pred_class = (y_train_pred >= threshold).astype(int)

y_test_class = (y_test >= threshold).astype(int)
y_test_pred_class = (y_test_pred >= threshold).astype(int)

print('Precision:', precision_score(y_train_class, y_train_pred_class))
print('Recall:', recall_score(y_train_class, y_train_pred_class))
print('F1 Score:', f1_score(y_train_class, y_train_pred_class))

print('Precision:', precision_score(y_test_class, y_test_pred_class))
print('Recall:', recall_score(y_test_class, y_test_pred_class))
print('F1 Score:', f1_score(y_test_class, y_test_pred_class))

train_results = pd.DataFrame({
    'Time': train_time_index,
    'y_train_True': y_train.values,
    'y_train_pred': y_train_pred
})

test_results = pd.DataFrame({
    'Time': test_time_index,
    'y_test_True': y_test.values,
    'y_test_pred': y_test_pred
})


train_results.to_csv('train_predictions.csv', index=False, encoding='utf-8-sig')
test_results.to_csv('test_predictions.csv', index=False, encoding='utf-8-sig')

